# SET News Service — Company News & Disclosures for All Stocks

Fetch news/disclosures from the SET news search API — **market-wide in one call** (every stock on SET/mai), in English or Thai.

**You will learn how to:**
- Fetch the latest trading day's company news for ALL stocks
- Filter by symbol, date window, and headline keyword
- Use the response helpers (`filter_today`, `filter_by_tag`, `filter_by_symbol`)
- Read Thai headlines and switch to the all-sources feed
- Access per-stock news via the unified `Stock` class

> ⚠️ **Two API gotchas** (live-verified): dates must be **dd/MM/yyyy** (ISO `yyyy-MM-dd` → HTTP 400 — the library validates eagerly and raises `InvalidDateError`), and `sourceId` values other than `company` are silently ignored (→ ALL sources). Pass `source_id=None` when you *want* all sources.

## 1. Setup

In [ ]:
from datetime import date, timedelta

from settfex.services.set import Stock, get_news

# In Jupyter, await works directly (no asyncio.run needed)

## 2. Latest company news — all stocks, one call

Without date filters the API returns the **latest trading day** window (~150–200 items). Default `source_id="company"` = company disclosures.

In [ ]:
news = await get_news()

print(f"Items: {news.count} (totalCount={news.total_count})")
for item in news.news_info_list[:8]:
    print(f"{item.news_datetime:%Y-%m-%d %H:%M}  {item.symbol:10s} {item.headline[:70]}")

## 3. Filter by symbol

Symbols are auto-uppercased (`'cpall'` → `'CPALL'`).

In [ ]:
# Pick a symbol that has news in the current window (first item's symbol)
symbol = news.news_info_list[0].symbol

symbol_news = await get_news(symbol=symbol.lower())
print(f"{symbol}: {symbol_news.count} item(s)")
for item in symbol_news.news_info_list:
    print(f"  {item.news_datetime:%Y-%m-%d %H:%M}  {item.headline[:70]}")

## 4. Date window

Pass `datetime.date`/`datetime` objects (converted to dd/MM/yyyy automatically) or dd/MM/yyyy strings. Keep windows modest — there is no pagination, so a 17-day window already returns ~2,800 items in one response.

In [ ]:
to_d = date.today()
from_d = to_d - timedelta(days=7)

week = await get_news(from_date=from_d, to_date=to_d)
print(f"Last 7 days: {week.count} items")

# Items per day
per_day: dict[str, int] = {}
for item in week.news_info_list:
    key = item.news_datetime.strftime("%Y-%m-%d")
    per_day[key] = per_day.get(key, 0) + 1
for day, n in sorted(per_day.items()):
    print(f"  {day}: {n}")

## 5. Headline keyword search

In [ ]:
dividend_news = await get_news(keyword="dividend")
print(f"Headlines matching 'dividend': {dividend_news.count}")
for item in dividend_news.news_info_list[:5]:
    print(f"  {item.symbol:10s} {item.headline[:70]}")

## 6. Response helpers

Observed tags: `''`, `financial-statement`, `nav`, `ca`, `set-releases`.

In [ ]:
print(f"Published today:       {len(news.filter_today())}")
print(f"Financial statements:  {len(news.filter_by_tag('financial-statement'))}")
print(f"NAV reports:           {len(news.filter_by_tag('nav'))}")
print(f"For {symbol}:          {len(news.filter_by_symbol(symbol))}")

## 7. Thai headlines

In [ ]:
thai_news = await get_news(lang="th")
print(f"Thai items: {thai_news.count}")
for item in thai_news.news_info_list[:5]:
    print(f"  {item.symbol:10s} {item.headline[:60]}")

## 8. All sources (superset)

`source_id=None` omits the parameter → SET-issued releases, funds, DW/TFEX rows are included on top of company disclosures.

In [ ]:
all_sources = await get_news(source_id=None)
print(f"All sources: {all_sources.count}  vs  company only: {news.count}")

extra_tags = {item.tag for item in all_sources.news_info_list} - {
    item.tag for item in news.news_info_list
}
print(f"Extra tags in the superset: {extra_tags}")

## 9. Per-stock access via the unified `Stock` class

In [ ]:
stock = Stock("CPALL")
cpall_news = await stock.get_news(from_date=from_d, to_date=to_d)
print(f"CPALL news in the last 7 days: {cpall_news.count}")
for item in cpall_news.news_info_list[:5]:
    print(f"  {item.news_datetime:%Y-%m-%d}  {item.headline[:70]}")

## 10. Export to pandas (optional: `pip install "settfex[dataframe]"`)

In [ ]:
try:
    import pandas as pd

    df = pd.DataFrame([item.model_dump() for item in week.news_info_list])
    df = df.sort_values("news_datetime", ascending=False)
    display(df[["news_datetime", "symbol", "headline", "tag", "url"]].head(10))
except ImportError:
    print('pandas not installed — pip install "settfex[dataframe]"')

## 11. Historical news & the rolling 5-year window

The endpoint serves a **rolling ~5-year history — exactly 1826 days** (5 × 365 + 1 leap day). A window whose `from_date` is older than `today − 1826 days` is rejected with **HTTP 400**, surfaced by the library as `FetchError` (**not** `InvalidDateError` — that one is only for malformed dd/MM/yyyy strings). The check is on the window's **start**, and the API does **not** clip to the allowed range: one day too old and the whole request fails.

> Live-verified 2026-07-20. The boundary is *rolling* — always `today − 1826 days`, never a fixed calendar date. Within the window, historical queries work exactly like the date-window query in §4.

In [ ]:
# Fetch a historical window well inside the 5-year limit (~1 year ago).
# Keep it small — there is no pagination, so a wide window returns everything at once.
hist_to = date.today() - timedelta(days=365)
hist_from = hist_to - timedelta(days=5)

hist = await get_news(from_date=hist_from, to_date=hist_to)
print(f"{hist_from} -> {hist_to}: {hist.count} items")

if hist.news_info_list:
    dts = [item.news_datetime for item in hist.news_info_list]
    print(f"  earliest: {min(dts):%Y-%m-%d %H:%M}")
    print(f"  latest:   {max(dts):%Y-%m-%d %H:%M}\n")
    for item in hist.news_info_list[:5]:
        print(f"  {item.news_datetime:%Y-%m-%d}  {item.symbol:10s} {item.headline[:60]}")

In [ ]:
from settfex.exceptions import FetchError

OLDEST_DAYS = 1826  # rolling 5-year window (= 5*365 + 1 leap day), live-verified 2026-07-20
oldest_ok = date.today() - timedelta(days=OLDEST_DAYS)
too_old = date.today() - timedelta(days=OLDEST_DAYS + 1)


async def try_from(from_d: date) -> None:
    """Probe a single-day window and report OK (served) vs rejected (too old)."""
    days_back = (date.today() - from_d).days
    try:
        res = await get_news(from_date=from_d, to_date=from_d)
        print(f"  {from_d} (today-{days_back}d): OK       -> {res.count} items")
    except FetchError as exc:
        # HTTP 400 for over-old windows surfaces as FetchError (NOT InvalidDateError)
        print(f"  {from_d} (today-{days_back}d): rejected -> {type(exc).__name__} (HTTP 400)")


print(f"Oldest servable date today: {oldest_ok}\n")
await try_from(oldest_ok)  # exactly the boundary -> OK
await try_from(too_old)    # one day too old   -> FetchError

## Use Cases

- **Disclosure monitoring**: poll `get_news()` intraday and alert on new items for your portfolio symbols (`filter_by_symbol`)
- **Earnings season tracking**: `filter_by_tag('financial-statement')` per day
- **Event-driven research**: date-window queries joined onto price history
- **AI/LLM pipelines**: `get_news()` is the flat tool-calling entry point; feed `headline` + `url` into a summarizer per symbol

**See also**: `docs/settfex/services/set/news.md` · Corporate Actions (05) for structured XD/XM events · Earnings Call (12) for OPPDAY transcripts